**Author**: Felipe Matheus  
**Start Date**: 12/06/2026  
**End Date**: --/06/2026  
**Purpose**: Experiment launcher ("control panel") for the annealing_iacs surrogate.

This notebook does NOT contain pipeline logic. All the logic (Model A -> OOF ->
NNLS -> Model B -> calibration -> metrics -> persistence) lives in
`src/modeling/Experiments.py`, which orchestrates the existing `Modeling` and
`Evaluation` helpers. Here you only:

1. Load and prepare the data (once).
2. Define a base `ExperimentConfig`.
3. Define the grid of variations you want to sweep.
4. Run and inspect the central `experiments_log.csv`.

Results layout on disk:
```
models/annealing_iacs/experiments/
    experiments_log.csv          <- 1 row per run (the "results spreadsheet")
    <tag>__<hash>/               <- 1 folder per run
        config.yaml
        model_a/   model_b/
        artifacts.pkl
        leaderboard_autogluon.csv
```

# 1. Setup

In [1]:
import logging
import os
import sys

import numpy as np
import pandas as pd

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.modeling.Modeling import Modeling
from src.modeling.Evaluation import Evaluation
from src.modeling.Experiments import ExperimentConfig, ExperimentRunner

from config.Variables import Variables

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

%load_ext autoreload
%autoreload 2

varv = Variables()
proc = Processing()
feng = FeatureEngineering()
modl = Modeling()
evla = Evaluation()
runr = ExperimentRunner(modl, evla, models_root=varv.PATHS.models)

c:\Users\fmfoa\Projects\uncertainty-aware-predictors\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2. Data (same preparation as annealing_iacs.ipynb, run once)

In [19]:
SCHEMA_DATE = "080626"
FILE_NAME = "dataset_annealing_iacs.csv"
FILE_NAME_SCHEMA_DATA = f"schema_annealing_essays_{SCHEMA_DATE}.csv"
FILE_NAME_VALIDATION_DATA = "Experimental Results Annealing V2 - CompiledResults.csv"

TARGET = "iacs_final"
ALL_FEATURES = ["purity", "iacs", "temperature", "time"]

TAG = "annealing-v3-test"
GRID = {
    "time_limit_a": [120, 300, 600],
    "weight_on_essay_rows": [1.0, 2.0],
    "features": [
        ("purity", "iacs", "temperature", "time"),
        ("iacs", "temperature", "time"),
    ],
}

In [ ]:
# ---- Schema (essay) data: explicit is_essay marker ----
df_raw_schema = pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME_SCHEMA_DATA))
df_schema = df_raw_schema[ALL_FEATURES + [TARGET]].dropna()
df_schema["is_essay"] = True

# ---- Literature data ----
df_raw = pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME))
df_float = proc.df_to_float(df_raw, drop_cols=["DOI"], ignore_columns=["material"])
df_labeled = feng.label_element(df_float).drop_duplicates()
df_with_masks = feng.add_ratio_mask_column(
    feng.add_ratio_mask_column(df_labeled, "grain_size"), "iacs",
)
df_lit = df_with_masks[df_with_masks.has_Cu == True][ALL_FEATURES + [TARGET]]
df_lit["is_essay"] = False

# ---- Validation data ----
df_val = proc.load_validation_data_chimie_paris(
    path=os.path.join(varv.PATHS.data_processed, FILE_NAME_VALIDATION_DATA)
)

assert df_lit[ALL_FEATURES + [TARGET]].isna().sum().sum() == 0, "NaNs in inputs"

# ---- Concat. weight_col is built INSIDE the runr from is_essay + config ----
df = pd.concat([df_schema, df_lit], ignore_index=True)
print(f"Dataset: {df.shape} | essays: {df.is_essay.sum()} | lit: {(~df.is_essay).sum()}")
df.head()

In [3]:
df_schema

,purity,iacs,temperature,time,iacs_final,is_essay
0,99.9,88.260,623.0,30.0,88.350000,True
1,99.9,88.260,623.0,60.0,88.655000,True
2,99.9,88.260,623.0,90.0,88.800000,True
3,99.9,99.825,573.0,30.0,102.340000,True
4,99.9,99.390,573.0,30.0,102.230000,True
5,99.9,99.390,588.0,30.0,101.346667,True
6,99.9,99.230,573.0,30.0,101.355000,True
7,99.9,98.720,573.0,30.0,101.480000,True
17,99.9,100.250,573.0,30.0,102.755000,True


In [4]:
df_val

,id,purity,initial_diameter,iacs,temperature,time,iacs_final
0,C1_250_30m_T1,99.95,1.2,98.47,523,30,101.08
1,C1_250_30m_T3,99.95,1.2,98.47,523,30,101.08
2,C1_250_60m_T1,99.95,1.2,98.47,523,60,100.82
3,C1_250_60m_T2,99.95,1.2,98.47,523,60,100.82
4,C1_250_90m_T1,99.95,1.2,98.47,523,90,101.22
5,C1_250_90m_T2,99.95,1.2,98.47,523,90,101.22
6,C1_250_30m_AQ_T1,99.95,1.2,98.47,523,30,100.95
7,C1_250_30m_AQ_T2,99.95,1.2,98.47,523,30,100.95
8,C1_300_30m_T1,99.95,1.2,98.47,573,30,102.03
9,C1_300_30m_T2,99.95,1.2,98.47,573,30,102.03


In [5]:
df

,purity,iacs,temperature,time,iacs_final,is_essay
0,99.9,88.260,623.0,30.0,88.350,True
1,99.9,88.260,623.0,60.0,88.655,True
2,99.9,88.260,623.0,90.0,88.800,True
3,99.9,99.825,573.0,30.0,102.340,True
4,99.9,99.390,573.0,30.0,102.230,True
...,...,...,...,...,...,...
91,99.0,88.000,673.0,30.0,97.000,False
92,99.0,87.000,473.0,30.0,89.000,False
93,99.0,87.000,573.0,30.0,92.000,False
94,99.0,87.000,673.0,30.0,95.000,False


# 3. Base config

In [6]:
base = ExperimentConfig(
    process="annealing_iacs",
    tag=TAG,
    target=TARGET,
    features=tuple(ALL_FEATURES),
    #     presets_a = "medium_quality"
    #     presets_b = "medium_quality"
    # everything else uses the defaults; override here if needed, e.g.:
    # time_limit_a=120, weight_on_essay_rows=1.0, use_shared_folds=False,
)
print(base.run_id)

annealing-v3-test__a793edd1


# 4. Single run (sanity check before any grid)

Always run the base config alone first. Then run it 2-3 more times with
`tag="annealing-v1-rep2"` etc. to measure run-to-run noise: AutoGluon under a
time budget is NOT deterministic, and at n~90 this noise is the floor below
which grid differences mean nothing.

In [7]:
df_val

,id,purity,initial_diameter,iacs,temperature,time,iacs_final
0,C1_250_30m_T1,99.95,1.2,98.47,523,30,101.08
1,C1_250_30m_T3,99.95,1.2,98.47,523,30,101.08
2,C1_250_60m_T1,99.95,1.2,98.47,523,60,100.82
3,C1_250_60m_T2,99.95,1.2,98.47,523,60,100.82
4,C1_250_90m_T1,99.95,1.2,98.47,523,90,101.22
5,C1_250_90m_T2,99.95,1.2,98.47,523,90,101.22
6,C1_250_30m_AQ_T1,99.95,1.2,98.47,523,30,100.95
7,C1_250_30m_AQ_T2,99.95,1.2,98.47,523,30,100.95
8,C1_300_30m_T1,99.95,1.2,98.47,573,30,102.03
9,C1_300_30m_T2,99.95,1.2,98.47,573,30,102.03


In [8]:
result = runr.run_experiment(df, base, df_val=df_val)
result["artifacts"]["metrics"]

2026-06-23 18:53:53,306 | INFO | src.modeling.Experiments | === Running annealing-v3-test__a793edd1 ===
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       12.97 GB / 31.57 GB (41.1%)
Disk Space Avail:   753.30 GB / 932.08 GB (80.8%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Values in column 'weight_col' used as sample weights instead of predictive features. Evaluation metrics will ignore sample weights, specify weight_evaluation=True to instead report weighted metrics.
Beginning AutoGluon training ... Time limit = 120s
AutoGluon will save models to "c:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v3-test\an

{'rmse': 3.996119956629481,
 'mae': 2.499519852185059,
 'mape': 3.187536363320045,
 'r2': 0.885691000328722}

In [9]:
result["artifacts"].keys()

dict_keys(['config', 'features', 'target', 'base_model_names', 'weights', 'nnls_recovery_ok', 'nnls_max_diff', 'variance_floor', 'recalibration_c', 'ood_ref', 'y_max', 'y_min', 'calibration_before', 'calibration_after', 'aleatoric_diagnostics', 'metrics', 'validation_metrics', 'dataset_hash'])

In [10]:
result.keys()

dict_keys(['cfg', 'run_dir', 'artifacts', 'log_row', 'predictor_a', 'predictor_b'])

# 5. Validation set

`df_val` rides along inside the runner: after training + calibration, the
calibrated predictive distribution is evaluated on it and `val_rmse`,
`val_mae`, `val_r2`, `val_cov_*` are appended to the SAME log row — so every
grid run carries fold (OOF) metrics AND validation metrics side by side.

Using `df_schema` as `df_val` is illustrative: those rows are inside the
training data, so val metrics are optimistic. Swap in any held-out
DataFrame with the same columns and nothing else changes.

In [20]:
result["artifacts"]["validation_metrics"]

{'rmse': 0.7936130294660124,
 'mae': 0.6655881888544555,
 'mape': 0.6596575497019993,
 'r2': -1.847867144716667,
 'coverage': {0.5: 1.0, 0.8: 1.0, 0.9: 1.0, 0.95: 1.0}}

In [21]:
result["artifacts"].keys()

dict_keys(['config', 'features', 'target', 'base_model_names', 'weights', 'nnls_recovery_ok', 'nnls_max_diff', 'variance_floor', 'recalibration_c', 'ood_ref', 'y_max', 'y_min', 'calibration_before', 'calibration_after', 'aleatoric_diagnostics', 'metrics', 'validation_metrics', 'dataset_hash'])

In [12]:
result.keys()

dict_keys(['cfg', 'run_dir', 'artifacts', 'log_row', 'predictor_a', 'predictor_b'])

In [13]:
result

{'cfg': ExperimentConfig(process='annealing_iacs', tag='annealing-v3-test', base_tag=None, target='iacs_final', features=('purity', 'iacs', 'temperature', 'time'), weight_on_essay_rows=1.0, presets_a='medium_quality', num_bag_folds_a=5, num_bag_sets_a=1, num_stack_levels_a=0, time_limit_a=120, presets_b='medium_quality', num_bag_folds_b=5, num_stack_levels_b=0, time_limit_b=60, use_weighted_variance=True, variance_floor_frac=0.01, recalibration_target_alpha=0.9, calibration_alphas=(0.5, 0.8, 0.9, 0.95), y_max=106.0, y_min=None, fold_seed=42, use_shared_folds=False),
 'run_dir': Path('../../models/annealing_iacs/experiments/annealing-v3-test/annealing-v3-test__a793edd1'),
 'artifacts': {'config': {'process': 'annealing_iacs',
   'tag': 'annealing-v3-test',
   'base_tag': None,
   'target': 'iacs_final',
   'features': ['purity', 'iacs', 'temperature', 'time'],
   'weight_on_essay_rows': 1.0,
   'presets_a': 'medium_quality',
   'num_bag_folds_a': 5,
   'num_bag_sets_a': 1,
   'num_stack

# 5. Grid

Keys are `ExperimentConfig` field names; values are lists of variants.
`features` variants must be tuples. Already-completed runs are skipped
(`force=True` to redo).

In [ ]:
log = runr.run_grid(df, base, GRID, df_val=df_val)   # 3 x 2 x 2 = 12 runs
log

2026-06-23 18:56:48,464 | INFO | src.modeling.Experiments | Grid: 12 runs over ['time_limit_a', 'weight_on_essay_rows', 'features']
2026-06-23 18:56:48,465 | INFO | src.modeling.Experiments | === Running annealing-v3-test__time_limit_a=120__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__a793edd1 ===
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       12.04 GB / 31.57 GB (38.2%)
Disk Space Avail:   752.42 GB / 932.08 GB (80.7%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Values in column 'weight_col' used as sample weights instead of predictive features. Evaluation metrics will ignore sample weights, specify weight_evaluation=True to instead

,run_id,timestamp,elapsed_s,n_rows,dataset_hash,model_dir,cfg_process,cfg_tag,cfg_base_tag,cfg_target,...,mean_sigma_aleat,n_val,val_rmse,val_mae,val_mape,val_r2,val_cov_0.5,val_cov_0.8,val_cov_0.9,val_cov_0.95
0,annealing-v3-test__a793edd1,2026-06-23T18:56:46,173.4,96,318f1408,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v3-test\annealing-v3-test__a793edd1,annealing_iacs,annealing-v3-test,NaN,iacs_final,...,1.19368,36,0.79361,0.66559,0.65966,-1.84787,1.0000,1.0,1.0,1.0
1,annealing-v3-test__time_limit_a=120__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__a793edd1,2026-06-23T18:59:54,185.9,96,318f1408,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v3-test\annealing-v3-test__time_limit_a=120__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__a793edd1,annealing_iacs,annealing-v3-test__time_limit_a=120__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time,annealing-v3-test,iacs_final,...,1.19368,36,0.79361,0.66559,0.65966,-1.84787,1.0000,1.0,1.0,1.0
2,annealing-v3-test__time_limit_a=120__weight_on_essay_rows=1.0__features=iacs-temperature-time__0489fac0,2026-06-23T19:03:05,191.1,96,c826d116,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v3-test\annealing-v3-test__time_limit_a=120__weight_on_essay_rows=1.0__features=iacs-temperature-time__0489fac0,annealing_iacs,annealing-v3-test__time_limit_a=120__weight_on_essay_rows=1.0__features=iacs-temperature-time,annealing-v3-test,iacs_final,...,1.45048,36,0.73241,0.53044,0.52609,-1.42558,0.9444,1.0,1.0,1.0
3,annealing-v3-test__time_limit_a=120__weight_on_essay_rows=2.0__features=purity-iacs-temperature-time__b1f6f7ef,2026-06-23T19:06:19,194.3,96,318f1408,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v3-test\annealing-v3-test__time_limit_a=120__weight_on_essay_rows=2.0__features=purity-iacs-temperature-time__b1f6f7ef,annealing_iacs,annealing-v3-test__time_limit_a=120__weight_on_essay_rows=2.0__features=purity-iacs-temperature-time,annealing-v3-test,iacs_final,...,1.28955,36,0.54369,0.46356,0.45892,-0.33662,1.0000,1.0,1.0,1.0
4,annealing-v3-test__time_limit_a=120__weight_on_essay_rows=2.0__features=iacs-temperature-time__4707eeaa,2026-06-23T19:08:45,145.6,96,c826d116,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v3-test\annealing-v3-test__time_limit_a=120__weight_on_essay_rows=2.0__features=iacs-temperature-time__4707eeaa,annealing_iacs,annealing-v3-test__time_limit_a=120__weight_on_essay_rows=2.0__features=iacs-temperature-time,annealing-v3-test,iacs_final,...,1.32237,36,0.60803,0.50622,0.50161,-0.67169,1.0000,1.0,1.0,1.0
5,annealing-v3-test__time_limit_a=300__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__b3191dc7,2026-06-23T19:12:23,217.6,96,318f1408,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v3-test\annealing-v3-test__time_limit_a=300__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__b3191dc7,annealing_iacs,annealing-v3-test__time_limit_a=300__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time,annealing-v3-test,iacs_final,...,1.19368,36,0.79361,0.66559,0.65966,-1.84787,1.0000,1.0,1.0,1.0
6,annealing-v3-test__time_limit_a=300__weight_on_essay_rows=1.0__features=iacs-temperature-time__3a9b1eb0,2026-06-23T19:15:50,207.0,96,c826d116,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_iacs\experiments\annealing-v3-test\annealing-v3-test__time_limit_a=300__weight_on_essay_rows=1.0__features=iacs-temperature-time__3a9b1eb0,annealing_iacs,annealing-v3-test__time_limit_a=300__weight_on_essay_rows=1.0__features=iacs-temperature-time,annealing-v3-test,iacs_final,...,1.26035,36,0.68084,0.54467,0.53978,-1.09603,1.0000,1.0,1.0,1.0
7,annealing-v3-test__time_limit_a=300__weight_on_essay_rows=2.

In [15]:
# import pickle
# from pathlib import Path
# import pandas as pd

# exp_dir = Path("../../models/annealing_iacs/experiments")
# rows = []
# for run_path in exp_dir.iterdir():
#     pkl = run_path / "artifacts.pkl"
#     if not pkl.exists():
#         continue
#     with open(pkl, "rb") as f:
#         art = pickle.load(f)
#     row = {"run_id": run_path.name}
#     row.update({f"cfg_{k}": v for k, v in art["config"].items()})
#     m = dict(art["metrics"]); m.pop("coverage", None)
#     row.update({k: v for k, v in m.items()})
#     cov = art["metrics"].get("coverage", {})
#     row.update({f"cov_{a}": c for a, c in cov.items()})
#     row["c_opt"] = art["recalibration_c"]
#     if art.get("validation_metrics"):
#         vm = dict(art["validation_metrics"]); vcov = vm.pop("coverage", {})
#         row.update({f"val_{k}": v for k, v in vm.items()})
#         row.update({f"val_cov_{a}": c for a, c in vcov.items()})
#     row["cfg_features"] = "|".join(art["features"])
#     rows.append(row)

# pd.DataFrame(rows).to_csv(exp_dir / "experiments_log.csv", index=False)
# print(f"Rebuilt log with {len(rows)} runs")

# 6. Inspect results

In [16]:
log = runr.load_log(base)

view_cols = [
    "run_id", "cfg_time_limit_a", "cfg_weight_on_essay_rows", "cfg_features",
    "rmse", "mae", "cov_0.9", "val_rmse", "val_mae", "val_cov_0.9", "c_opt", "pct_truncated_aleat",
    "mean_sigma_epist", "mean_sigma_aleat", "elapsed_s",
]
log[[c for c in view_cols if c in log.columns]].sort_values("rmse")

,run_id,cfg_time_limit_a,cfg_weight_on_essay_rows,cfg_features,rmse,mae,cov_0.9,val_rmse,val_mae,val_cov_0.9,c_opt,pct_truncated_aleat,mean_sigma_epist,mean_sigma_aleat,elapsed_s
7,annealing-v3-test__time_limit_a=300__weight_on_essay_rows=2.0__features=purity-iacs-temperature-time__256c2ab9,300,2.0,purity|iacs|temperature|time,3.94129,2.50567,0.9062,0.73800,0.62682,1.0,1.1902,39.58,1.61382,1.37757,209.9
11,annealing-v3-test__time_limit_a=600__weight_on_essay_rows=2.0__features=purity-iacs-temperature-time__6937b788,600,2.0,purity|iacs|temperature|time,3.94129,2.50567,0.9062,0.73800,0.62682,1.0,1.1902,39.58,1.61382,1.37757,206.1
5,annealing-v3-test__time_limit_a=300__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__b3191dc7,300,1.0,purity|iacs|temperature|time,3.99612,2.49952,0.9062,0.79361,0.66559,1.0,1.4249,47.92,1.67132,1.19368,217.6
0,annealing-v3-test__a793edd1,120,1.0,purity|iacs|temperature|time,3.99612,2.49952,0.9062,0.79361,0.66559,1.0,1.4249,47.92,1.67132,1.19368,173.4
9,annealing-v3-test__time_limit_a=600__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__d46f25b3,600,1.0,purity|iacs|temperature|time,3.99612,2.49952,0.9062,0.79361,0.66559,1.0,1.4249,47.92,1.67132,1.19368,228.6
1,annealing-v3-test__time_limit_a=120__weight_on_essay_rows=1.0__features=purity-iacs-temperature-time__a793edd1,120,1.0,purity|iacs|temperature|time,3.99612,2.49952,0.9062,0.79361,0.66559,1.0,1.4249,47.92,1.67132,1.19368,185.9
3,annealing-v3-test__time_limit_a=120__weight_on_essay_rows=2.0__features=purity-iacs-temperature-time__b1f6f7ef,120,2.0,purity|iacs|temperature|time,4.01810,2.59646,0.9062,0.54369,0.46356,1.0,1.4549,33.33,1.49009,1.28955,194.3
4,annealing-v3-test__time_limit_a=120__weight_on_essay_rows=2.0__features=iacs-temperature-time__4707eeaa,120,2.0,iacs|temperature|time,4.42644,2.92060,0.8958,0.60803,0.50622,1.0,2.3585,41.67,1.39880,1.32237,145.6
8,annealing-v3-test__time_limit_a=300__weight_on_essay_rows=2.0__features=iacs-temperature-time__333b70e4,300,2.0,iacs|temperature|time,4.42644,2.92060,0.8958,0.60803,0.50622,1.0,2.3585,41.67,1.39880,1.32237,143.7
12,annealing-v3-test__time_limit_a=600__weight_on_essay_rows=2.0__features=iacs-temperature-time__9a750067,600,2.0,iacs|temperature|time,4.42644,2.92060,0.8958,0.60803,0.50622,1.0,2.3585,41.67,1.39880,1.32237,133.6


In [17]:
# Quick pivot: effect of one knob, marginalised over the others.
# Remember: compare against run-to-run noise (Section 4) before concluding.
log.groupby("cfg_time_limit_a")[["rmse", "mae", "cov_0.9"]].agg(["mean", "std"])

rmse                 mae            cov_0.9          
                      mean       std      mean       std     mean       std
cfg_time_limit_a                                                           
120               4.185746  0.250856  2.688520  0.218181  0.90204  0.005696
300               4.209148  0.279181  2.722402  0.254437  0.90100  0.006004
600               4.209148  0.279181  2.722402  0.254437  0.90100  0.006004

# 7. Load a winner for deployment / further analysis

Each run folder is self-contained: predictors + artifacts.pkl with weights,
`recalibration_c`, calibration tables.

In [ ]:
import pickle
from pathlib import Path
from autogluon.tabular import TabularPredictor

# log already carries the resolved absolute path, so use it directly
best_row = log.sort_values("rmse").iloc[0]
RUN_ID = best_row["run_id"]
run_dir = Path(best_row["model_dir"])   # <-- absolute path, base_tag included

with open(run_dir / "artifacts.pkl", "rb") as f:
    art = pickle.load(f)
predictor_a = TabularPredictor.load(str(run_dir / "model_a"))
predictor_b = TabularPredictor.load(str(run_dir / "model_b"))

print(RUN_ID)
art["calibration_after"]

annealing-v3-test__time_limit_a=300__weight_on_essay_rows=2.0__features=purity-iacs-temperature-time__256c2ab9


,alpha,empirical_coverage,gap
0,0.50,0.479167,-0.020833
1,0.80,0.791667,-0.008333
2,0.90,0.906250,0.006250
3,0.95,0.906250,-0.043750


In [ ]:
run_dir

Path('../../models/annealing_iacs/experiments/annealing-v3-test__time_limit_a=300__weight_on_essay_rows=2.0__features=purity-iacs-temperature-time__256c2ab9')

In [ ]:
run_dir